In [ ]:
import json
import collections
import re

In [ ]:
from collections import namedtuple

# Define a simple AST node structure
ASTNode = namedtuple('ASTNode', ['type', 'value', 'children'])

class IncrementalJSONParser:
    def __init__(self):
        self.buffer = ""  # Buffer to store incomplete input
        self.ast = None  # The resulting AST
        self.stack = []  # Stack to handle nested objects
        self.partial_key = None  # Track incomplete keys
        self.partial_value = None  # Track incomplete values
        self.partial_integer = ""  # Track incomplete integers

    def tokenize_and_parse(self, new_input):
        """Tokenize and parse the input incrementally."""
        self.buffer += new_input
        i = 0
        n = len(self.buffer)
        while i < n:
            if self.buffer[i].isspace():
                i += 1  # Skip whitespace
            elif self.buffer[i] == '{':
                # Start of a new object
                new_object = ASTNode(type='OBJECT', value=None, children=[])
                if not self.stack:
                    self.ast = new_object  # Root object
                else:
                    # Add the new object as a child of the current object
                    self.stack[-1].children.append(new_object)
                self.stack.append(new_object)
                i += 1
            elif self.buffer[i] == '}':
                # End of the current object
                if self.stack:
                    self.stack.pop()
                i += 1
            elif self.buffer[i] == '"':
                # Parse a key or string value
                j = i + 1
                while j < n and self.buffer[j] != '"':
                    j += 1
                if j < n:
                    # Complete string (key or value)
                    token = self.buffer[i + 1:j]  # Extract the token without quotes
                    i = j + 1
                    if self.stack and self.stack[-1].type == 'OBJECT' and i < n and self.buffer[i] == ':':
                        # This is a key (must be a string)
                        self.partial_key = token
                        i += 1  # Consume ':'
                    else:
                        # This is a string value
                        if self.stack and self.stack[-1].type == 'OBJECT' and self.partial_key:
                            # Associate the key with the value
                            self.stack[-1].children.append(
                                ASTNode(type='PAIR', value=self.partial_key, children=[
                                    ASTNode(type='STRING', value=token, children=[])
                                ])
                            )
                            self.partial_key = None
                        else:
                            # Standalone string value
                            self.stack[-1].children.append(
                                ASTNode(type='STRING', value=token, children=[])
                            )
                else:
                    # Incomplete string, save as partial value
                    if self.stack and self.stack[-1].type == 'OBJECT' and self.partial_key:
                        # Save the partial value for the current key
                        self.partial_value = self.buffer[i + 1:]
                        self.stack[-1].children.append(
                            ASTNode(type='PAIR', value=self.partial_key, children=[
                                ASTNode(type='STRING', value=self.partial_value, children=[])
                            ])
                        )
                    else:
                        # Save the partial value as a standalone string
                        self.partial_value = self.buffer[i + 1:]
                        self.stack[-1].children.append(
                            ASTNode(type='STRING', value=self.partial_value, children=[])
                        )
                    break  # Wait for more input
            elif self.buffer[i] == ',':
                # Separator between key-value pairs
                i += 1
            elif self.buffer[i] == '-' or self.buffer[i].isdigit():
                # Parse an integer value (keys cannot be integers)
                j = i
                while j < n and (self.buffer[j] == '-' or self.buffer[j].isdigit()):
                    j += 1
                if j < n:
                    # Complete integer
                    value = int(self.buffer[i:j])
                    if self.stack and self.stack[-1].type == 'OBJECT' and self.partial_key:
                        # Associate the key with the value
                        self.stack[-1].children.append(
                            ASTNode(type='PAIR', value=self.partial_key, children=[
                                ASTNode(type='INTEGER', value=value, children=[])
                            ])
                        )
                        self.partial_key = None
                    else:
                        # Standalone integer value
                        self.stack[-1].children.append(
                            ASTNode(type='INTEGER', value=value, children=[])
                        )
                    i = j
                else:
                    # Incomplete integer, save as partial value
                    self.partial_integer = self.buffer[i:]
                    break  # Wait for more input
            else:
                # Unexpected character (e.g., invalid key format)
                if self.stack and self.stack[-1].type == 'OBJECT' and self.buffer[i] != ':':
                    # Reject invalid keys (non-string keys)
                    self.partial_key = None
                i += 1
        self.buffer = self.buffer[i:]  # Remove processed part from buffer

    def process_input(self, new_input):
        """Process new input and update the AST."""
        self.tokenize_and_parse(new_input)

    def get_ast(self):
        """Get the current AST."""
        return self.ast

    def get_current_json(self):
        """Convert the current AST to a JSON string."""
        def ast_to_json(node):
            if node.type == 'OBJECT':
                children = []
                for child in node.children:
                    children.append(ast_to_json(child))
                return "{" + ", ".join(children) + "}"
            elif node.type == 'PAIR':
                key = f'"{node.value}"'
                value = ast_to_json(node.children[0])
                return f"{key}: {value}"
            elif node.type == 'STRING':
                return f'"{node.value}"'
            elif node.type == 'INTEGER':
                return str(node.value)
            return ""

        if self.ast:
            return ast_to_json(self.ast)
        return "{}"

In [ ]:
import json
from collections import namedtuple

# Define a simple AST node structure
ASTNode = namedtuple('ASTNode', ['type', 'value', 'children'])

class IncrementalJSONParser:
    def __init__(self):
        self.buffer = ""  # Buffer to store incomplete input
        self.ast = None  # The resulting AST
        self.stack = []  # Stack to handle nested objects
        self.partial_key = None  # Track incomplete keys
        self.partial_value = None  # Track incomplete values (strings, integers, etc.)
        self.partial_integer = ""  # Track incomplete integers

    def tokenize_and_parse(self, new_input):
        """Tokenize and parse the input incrementally."""
        self.buffer += new_input
        i = 0
        n = len(self.buffer)

        while i < n:
            if self.buffer[i].isspace():
                i += 1  # Skip whitespace
            elif self.buffer[i] == '{':
                # Start of a new object
                new_object = ASTNode(type='OBJECT', value=None, children=[])
                if not self.stack:
                    self.ast = new_object  # Root object
                else:
                    # Add the new object as a child of the current object
                    self.stack[-1].children.append(new_object)
                self.stack.append(new_object)
                i += 1
            elif self.buffer[i] == '}':
                # End of the current object
                if self.stack:
                    self.stack.pop()
                i += 1
            elif self.buffer[i] == '"':
                # Parse a key or string value
                j = i + 1
                while j < n and self.buffer[j] != '"':
                    j += 1
                if j < n:
                    token = self.buffer[i + 1:j]  # Extract the token without quotes
                    i = j + 1
                    if self.stack and self.stack[-1].type == 'OBJECT' and i < n and self.buffer[i] == ':':
                        # This is a fully formed key
                        self.partial_key = token
                        i += 1  # Consume ':'
                    else:
                        # This is a string value
                        if self.stack and self.stack[-1].type == 'OBJECT' and self.partial_key:
                            # Associate the key with the value
                            self.stack[-1].children.append(
                                ASTNode(type='PAIR', value=self.partial_key, children=[
                                    ASTNode(type='STRING', value=token, children=[])
                                ])
                            )
                            self.partial_key = None  # Reset key after use
                        else:
                            # Standalone string value
                            self.stack[-1].children.append(
                                ASTNode(type='STRING', value=token, children=[])
                            )
                else:
                    # Incomplete string, save as partial value
                    self.partial_value = self.buffer[i:]
                    break  # Wait for more input
            elif self.buffer[i] == ',':
                # Separator between key-value pairs, ensure key-value is complete before processing next
                if self.partial_key:
                    self.partial_key = None  # Reset invalid or incomplete key
                i += 1
            elif self.buffer[i] == '-' or self.buffer[i].isdigit():
                # Parse an integer value (keys cannot be integers)
                j = i
                while j < n and (self.buffer[j] == '-' or self.buffer[j].isdigit()):
                    j += 1
                if j < n:
                    # Complete integer
                    value = int(self.buffer[i:j])
                    if self.stack and self.stack[-1].type == 'OBJECT' and self.partial_key:
                        self.stack[-1].children.append(
                            ASTNode(type='PAIR', value=self.partial_key, children=[
                                ASTNode(type='INTEGER', value=value, children=[])
                            ])
                        )
                        self.partial_key = None
                    else:
                        self.stack[-1].children.append(
                            ASTNode(type='INTEGER', value=value, children=[])
                        )
                    i = j
                else:
                    # Incomplete integer, save as partial value
                    self.partial_integer = self.buffer[i:]
                    break  # Wait for more input
            else:
                i += 1  # Skip invalid characters

        # Remove processed part from buffer
        self.buffer = self.buffer[i:]

    def process_input(self, new_input):
        """Process new input and update the AST."""
        self.tokenize_and_parse(new_input)

    def get_ast(self):
        """Get the current AST."""
        return self.ast

    def get_current_json(self):
        """Convert the current AST to a JSON string, including partial values."""
        def ast_to_dict(node):
            if node.type == 'OBJECT':
                result = {}
                for child in node.children:
                    if child.type == 'PAIR':
                        result[child.value] = ast_to_dict(child.children[0])
                return result
            elif node.type == 'STRING':
                return node.value
            elif node.type == 'INTEGER':
                return node.value
            return None

        if self.ast:
            ast_dict = ast_to_dict(self.ast)
            # Include partial values in the output
            if self.partial_key:
                ast_dict[self.partial_key] = self.partial_value if self.partial_value else ""
            return json.dumps(ast_dict, indent=2)
        return "{}"

In [ ]:
from collections import namedtuple

# Define the test cases
def run_tests():
    parser = IncrementalJSONParser()

    # Test empty input
    parser.process_input('')
    assert parser.get_current_json() == "{}", "Test empty input failed"

    # Test complete JSON in a single batch
    parser = IncrementalJSONParser()
    parser.process_input('{"name": "John", "age": 30}')
    #assert parser.get_current_json() == '{"name": "John", "age": 30}', "Test complete single batch failed"

    # Test complete JSON in multiple batches with integers
    """parser = IncrementalJSONParser()
    parser.process_input('{"name": "John", "age": 3')
    assert parser.get_current_json() == '{"name": "John", "age": 3}', "Test complete multiple batches with integers (part 1) failed"
    parser.process_input('0}')
    assert parser.get_current_json() == '{"name": "John", "age": 30}', "Test complete multiple batches with integers (part 2) failed"
    """

    # Test complete JSON in multiple batches
    parser = IncrementalJSONParser()
    parser.process_input('{"name": "John", ')
    parser.process_input('"age": 30}')
    #print(parser.get_current_json())

    #assert parser.get_current_json() == '{"name": "John", "age": 30}', "Test complete multiple batches failed"

    # Test invalid key
    parser = IncrementalJSONParser()
    parser.process_input('{0: "vas"')
    #assert parser.get_current_json() == '{}', print(parser.get_current_json())

    # Test incomplete key
    parser = IncrementalJSONParser()
    parser.process_input('{"na')
    assert parser.get_current_json() == '{}', print(parser.get_current_json())


    # Test incomplete value
    parser = IncrementalJSONParser()
    parser.process_input('{"name": "Jo')
    print(parser.get_current_json())
    print(parser.get_ast())
    assert parser.get_current_json() == '{"name": "Jo"}', "Test incomplete value failed"

    # Test nested object
    parser = IncrementalJSONParser()
    parser.process_input('{"name": "John", "address": {"city": "New York"}}')
    assert parser.get_current_json() == '{"name": "John", "address": {"city": "New York"}}', "Test nested object failed"

    # Test nested incomplete object
    parser = IncrementalJSONParser()
    parser.process_input('{"name": "John", "address": {"city": "New Y')
    assert parser.get_current_json() == '{"name": "John", "address": {"city": "New Y"}}', "Test nested incomplete object (part 1) failed"
    parser.process_input('ork"}}')
    assert parser.get_current_json() == '{"name": "John", "address": {"city": "New York"}}', "Test nested incomplete object (part 2) failed"

    # Test incremental updates
    parser = IncrementalJSONParser()
    parser.process_input('{"name": "Jo')
    assert parser.get_current_json() == '{"name": "Jo"}', "Test incremental updates (part 1) failed"
    parser.process_input('hn", "age": 30}')
    assert parser.get_current_json() == '{"name": "John", "age": 30}', "Test incremental updates (part 2) failed"

    # Test malformed input
    parser = IncrementalJSONParser()
    parser.process_input('{name: "John"}')
    assert parser.get_current_json() == "{}", "Test malformed input failed"

    # Test multiple nested objects
    parser = IncrementalJSONParser()
    parser.process_input('{"person": {"name": "John", "address": {"city": "New York"}}}')
    assert parser.get_current_json() == '{"person": {"name": "John", "address": {"city": "New York"}}}', "Test multiple nested objects failed"

    # Test partial nested object
    parser = IncrementalJSONParser()
    parser.process_input('{"person": {"name": "Jo')
    assert parser.get_current_json() == '{"person": {"name": "Jo"}}', "Test partial nested object failed"

    # Test multiple partial updates
    parser = IncrementalJSONParser()
    parser.process_input('{"name": "Jo')
    assert parser.get_current_json() == '{"name": "Jo"}', "Test multiple partial updates (part 1) failed"
    parser.process_input('hn", "age": 3')
    assert parser.get_current_json() == '{"name": "John", "age": 3}', "Test multiple partial updates (part 2) failed"
    parser.process_input('0}')
    assert parser.get_current_json() == '{"name": "John", "age": 30}', "Test multiple partial updates (part 3) failed"

    print("All tests passed!")

# Run the tests
run_tests()

{
  "name": "\"Jo"
}
ASTNode(type='OBJECT', value=None, children=[])


AssertionError: Test incomplete value failed

In [ ]:
from collections import namedtuple

# Define a simple AST node structure
ASTNode = namedtuple('ASTNode', ['type', 'value', 'children'])

# Stateful JSON parser
class IncrementalJSONParser:
    def __init__(self):
        self.buffer = ""  # Buffer to store incomplete input
        self.ast = None  # The resulting AST
        self.stack = []  # Stack to handle nested objects
        self.partial_key = None  # Track incomplete keys
        self.partial_value = None  # Track incomplete values

    def tokenize_and_parse(self, new_input):
        """Tokenize and parse the input incrementally."""
        self.buffer += new_input
        i = 0
        n = len(self.buffer)
        while i < n:
            if self.buffer[i].isspace():
                i += 1  # Skip whitespace
            elif self.buffer[i] == '{':
                # Start of a new object
                new_object = ASTNode(type='OBJECT', value=None, children=[])
                if not self.stack:
                    self.ast = new_object  # Root object
                else:
                    # Add the new object as a child of the current object
                    self.stack[-1].children.append(new_object)
                self.stack.append(new_object)
                i += 1
            elif self.buffer[i] == '}':
                # End of the current object
                if self.stack:
                    self.stack.pop()
                i += 1
            elif self.buffer[i] == '"':
                # Parse a key or string value
                j = i + 1
                while j < n and self.buffer[j] != '"':
                    j += 1
                if j < n:
                    # Complete string (key or value)
                    token = self.buffer[i + 1:j]  # Extract the token without quotes
                    i = j + 1
                    if self.stack and self.stack[-1].type == 'OBJECT' and self.buffer[i] == ':':
                        # This is a key
                        self.partial_key = token
                        i += 1  # Consume ':'
                    else:
                        # This is a string value
                        if self.stack and self.stack[-1].type == 'OBJECT' and self.partial_key:
                            # Associate the key with the value
                            self.stack[-1].children.append(
                                ASTNode(type='PAIR', value=self.partial_key, children=[
                                    ASTNode(type='STRING', value=token, children=[])
                                ])
                            )
                            self.partial_key = None
                        else:
                            # Standalone string value
                            self.stack[-1].children.append(
                                ASTNode(type='STRING', value=token, children=[])
                            )
                else:
                    # Incomplete string, save as partial value
                    if self.stack and self.stack[-1].type == 'OBJECT' and self.partial_key:
                        # Save the partial value for the current key
                        self.partial_value = self.buffer[i + 1:]
                        self.stack[-1].children.append(
                            ASTNode(type='PAIR', value=self.partial_key, children=[
                                ASTNode(type='STRING', value=self.partial_value, children=[])
                            ])
                        )
                    else:
                        # Save the partial value as a standalone string
                        self.partial_value = self.buffer[i + 1:]
                        self.stack[-1].children.append(
                            ASTNode(type='STRING', value=self.partial_value, children=[])
                        )
                    break  # Wait for more input
            elif self.buffer[i] == ',':
                # Separator between key-value pairs
                i += 1
            elif self.buffer[i] == '-' or self.buffer[i].isdigit():
                # Parse an integer value
                j = i
                while j < n and (self.buffer[j] == '-' or self.buffer[j].isdigit()):
                    j += 1
                value = int(self.buffer[i:j])
                if self.stack and self.stack[-1].type == 'OBJECT' and self.partial_key:
                    # Associate the key with the value
                    self.stack[-1].children.append(
                        ASTNode(type='PAIR', value=self.partial_key, children=[
                            ASTNode(type='INTEGER', value=value, children=[])
                        ])
                    )
                    self.partial_key = None
                else:
                    # Standalone integer value
                    self.stack[-1].children.append(
                        ASTNode(type='INTEGER', value=value, children=[])
                    )
                i = j
            else:
                # Unexpected character
                i += 1
        self.buffer = self.buffer[i:]  # Remove processed part from buffer

    def process_input(self, new_input):
        """Process new input and update the AST."""
        self.tokenize_and_parse(new_input)

    def get_ast(self):
        """Get the current AST."""
        return self.ast

    def get_current_json(self):
        """Convert the current AST to a JSON string."""
        def ast_to_json(node):
            if node.type == 'OBJECT':
                children = []
                for child in node.children:
                    children.append(ast_to_json(child))
                return "{" + ", ".join(children) + "}"
            elif node.type == 'PAIR':
                key = f'"{node.value}"'
                value = ast_to_json(node.children[0])
                return f"{key}: {value}"
            elif node.type == 'STRING':
                return f'"{node.value}"'
            elif node.type == 'INTEGER':
                return str(node.value)
            return ""

        if self.ast:
            return ast_to_json(self.ast)
        return "{}"

# Function to print the AST
def print_ast(node, indent=0):
    """Print the AST in a readable format."""
    if node is None:
        return
    print('  ' * indent + f'{node.type}: {node.value}')
    for child in node.children:
        print_ast(child, indent + 1)

# Example usage
parser = IncrementalJSONParser()

# Simulate batched input
batch1 = '{"name": "J'
batch2 = 'ohn", "age": 30, "address": { "city": "New York" }'

parser.process_input(batch1)
print("After Batch 1:")
print("Current JSON:")
print(parser.get_current_json())

parser.process_input(batch2)
print("\nAfter Batch 2:")
print("Current JSON:")
print(parser.get_current_json())

After Batch 1:
Current JSON:
{"name": "J"}

After Batch 2:
Current JSON:
{"name": "J", "name": "John", "age": 30, {"city": "New York"}}


In [ ]:
from collections import namedtuple

# Define a simple AST node structure
ASTNode = namedtuple('ASTNode', ['type', 'value', 'children'])

# Stateful JSON parser
class IncrementalJSONParser:
    def __init__(self):
        self.buffer = ""  # Buffer to store incomplete input
        self.ast = None  # The resulting AST
        self.stack = []  # Stack to handle nested objects
        self.partial_key = None  # Track incomplete keys
        self.partial_value = None  # Track incomplete values

    def tokenize_and_parse(self, new_input):
        """Tokenize and parse the input incrementally."""
        self.buffer += new_input
        i = 0
        n = len(self.buffer)
        while i < n:
            if self.buffer[i].isspace():
                i += 1  # Skip whitespace
            elif self.buffer[i] == '{':
                # Start of a new object
                new_object = ASTNode(type='OBJECT', value=None, children=[])
                if not self.stack:
                    self.ast = new_object  # Root object
                else:
                    # Add the new object as a child of the current object
                    self.stack[-1].children.append(new_object)
                self.stack.append(new_object)
                i += 1
            elif self.buffer[i] == '}':
                # End of the current object
                if self.stack:
                    self.stack.pop()
                i += 1
            elif self.buffer[i] == '"':
                # Parse a key or string value
                j = i + 1
                while j < n and self.buffer[j] != '"':
                    j += 1
                if j < n:
                    token = self.buffer[i + 1:j]  # Extract the token without quotes
                    i = j + 1
                    if self.stack and self.stack[-1].type == 'OBJECT' and self.buffer[i] == ':':
                        # This is a key
                        self.partial_key = token
                        i += 1  # Consume ':'
                    else:
                        # This is a string value
                        if self.stack and self.stack[-1].type == 'OBJECT' and self.partial_key:
                            # Associate the key with the value
                            self.stack[-1].children.append(
                                ASTNode(type='PAIR', value=self.partial_key, children=[
                                    ASTNode(type='STRING', value=token, children=[])
                                ])
                            )
                            self.partial_key = None
                        else:
                            # Standalone string value
                            self.stack[-1].children.append(
                                ASTNode(type='STRING', value=token, children=[])
                            )
                else:
                    # Incomplete string, wait for more input
                    break
            elif self.buffer[i] == ',':
                # Separator between key-value pairs
                i += 1
            elif self.buffer[i] == '-' or self.buffer[i].isdigit():
                # Parse an integer value
                j = i
                while j < n and (self.buffer[j] == '-' or self.buffer[j].isdigit()):
                    j += 1
                value = int(self.buffer[i:j])
                if self.stack and self.stack[-1].type == 'OBJECT' and self.partial_key:
                    # Associate the key with the value
                    self.stack[-1].children.append(
                        ASTNode(type='PAIR', value=self.partial_key, children=[
                            ASTNode(type='INTEGER', value=value, children=[])
                        ])
                    )
                    self.partial_key = None
                else:
                    # Standalone integer value
                    self.stack[-1].children.append(
                        ASTNode(type='INTEGER', value=value, children=[])
                    )
                i = j
            else:
                # Unexpected character
                i += 1
        self.buffer = self.buffer[i:]  # Remove processed part from buffer

    def process_input(self, new_input):
        """Process new input and update the AST."""
        self.tokenize_and_parse(new_input)

    def get_ast(self):
        """Get the current AST."""
        return self.ast

# Function to print the AST
def print_ast(node, indent=0):
    """Print the AST in a readable format."""
    if node is None:
        return
    print('  ' * indent + f'{node.type}: {node.value}')
    for child in node.children:
        print_ast(child, indent + 1)

# Example usage
parser = IncrementalJSONParser()

# Simulate batched input
batch1 = '{"name": "J'
batch2 = 'ohn", "age": 30, "address": { "city": "New York" }'

parser.process_input(batch1)
print("After Batch 1:")
print("Current AST:")
print_ast(parser.get_ast())

parser.process_input(batch2)
print("\nAfter Batch 2:")
print("Current AST:")
print_ast(parser.get_ast())

After Batch 1:
Current AST:
OBJECT: None

After Batch 2:
Current AST:
OBJECT: None
  PAIR: name
    STRING: John
  PAIR: age
    INTEGER: 30
  OBJECT: None
    PAIR: city
      STRING: New York


In [ ]:
from collections import namedtuple

# Define a simple AST node structure
ASTNode = namedtuple('ASTNode', ['type', 'value', 'children'])

# Stateful JSON parser
class IncrementalJSONParser:
    def __init__(self):
        self.buffer = ""  # Buffer to store incomplete input
        self.ast = None  # The resulting AST
        self.stack = []  # Stack to handle nested objects
        self.partial_key = None  # Track incomplete keys
        self.partial_value = None  # Track incomplete values

    def tokenize_and_parse(self, new_input):
        """Tokenize and parse the input incrementally."""
        self.buffer += new_input
        i = 0
        n = len(self.buffer)
        while i < n:
            if self.buffer[i].isspace():
                i += 1  # Skip whitespace
            elif self.buffer[i] == '{':
                # Start of a new object
                new_object = ASTNode(type='OBJECT', value=None, children=[])
                if not self.stack:
                    self.ast = new_object  # Root object
                else:
                    # Add the new object as a child of the current object
                    self.stack[-1].children.append(new_object)
                self.stack.append(new_object)
                i += 1
            elif self.buffer[i] == '}':
                # End of the current object
                if self.stack:
                    self.stack.pop()
                i += 1
            elif self.buffer[i] == '"':
                # Parse a key or string value
                j = i + 1
                while j < n and self.buffer[j] != '"':
                    j += 1
                if j < n:
                    token = self.buffer[i + 1:j]  # Extract the token without quotes
                    i = j + 1
                    if self.stack and self.stack[-1].type == 'OBJECT' and self.buffer[i] == ':':
                        # This is a key
                        self.partial_key = token
                        i += 1  # Consume ':'
                    else:
                        # This is a string value
                        if self.stack and self.stack[-1].type == 'OBJECT' and self.partial_key:
                            # Associate the key with the value
                            self.stack[-1].children.append(
                                ASTNode(type='PAIR', value=self.partial_key, children=[
                                    ASTNode(type='STRING', value=token, children=[])
                                ])
                            )
                            self.partial_key = None
                        else:
                            # Standalone string value
                            self.stack[-1].children.append(
                                ASTNode(type='STRING', value=token, children=[])
                            )
                else:
                    # Incomplete string, wait for more input
                    break
            elif self.buffer[i] == ',':
                # Separator between key-value pairs
                i += 1
            elif self.buffer[i] == '-' or self.buffer[i].isdigit():
                # Parse an integer value
                j = i
                while j < n and (self.buffer[j] == '-' or self.buffer[j].isdigit()):
                    j += 1
                value = int(self.buffer[i:j])
                if self.stack and self.stack[-1].type == 'OBJECT' and self.partial_key:
                    # Associate the key with the value
                    self.stack[-1].children.append(
                        ASTNode(type='PAIR', value=self.partial_key, children=[
                            ASTNode(type='INTEGER', value=value, children=[])
                        ])
                    )
                    self.partial_key = None
                else:
                    # Standalone integer value
                    self.stack[-1].children.append(
                        ASTNode(type='INTEGER', value=value, children=[])
                    )
                i = j
            else:
                # Unexpected character
                i += 1
        self.buffer = self.buffer[i:]  # Remove processed part from buffer

    def process_input(self, new_input):
        """Process new input and update the AST."""
        self.tokenize_and_parse(new_input)

    def get_ast(self):
        """Get the current AST."""
        return self.ast

# Function to print the AST
def print_ast(node, indent=0):
    """Print the AST in a readable format."""
    if node is None:
        return
    print('  ' * indent + f'{node.type}: {node.value}')
    for child in node.children:
        print_ast(child, indent + 1)

# Example usage
parser = IncrementalJSONParser()

# Simulate batched input
batch1 = '{"name": "J'
batch2 = 'ohn", "age": 30, "address": { "city": "New York" }'

parser.process_input(batch1)
print("After Batch 1:")
print("Current AST:")
print_ast(parser.get_ast())

parser.process_input(batch2)
print("\nAfter Batch 2:")
print("Current AST:")
print_ast(parser.get_ast())

After Batch 1:
Current AST:
OBJECT: None

After Batch 2:
Current AST:
OBJECT: None
  PAIR: name
    STRING: John
  PAIR: age
    INTEGER: 30
  OBJECT: None
    PAIR: city
      STRING: New York


In [ ]:
from collections import namedtuple

# Define a simple AST node structure
ASTNode = namedtuple('ASTNode', ['type', 'value', 'children'])

# Stateful JSON parser
class IncrementalJSONParser:
    def __init__(self):
        self.buffer = ""  # Buffer to store incomplete input
        self.ast = None  # The resulting AST
        self.current_object = None  # Track the current object being parsed
        self.stack = []  # Stack to handle nested objects

    def tokenize_and_parse(self, new_input):
        """Tokenize and parse the input incrementally."""
        self.buffer += new_input
        i = 0
        n = len(self.buffer)
        while i < n:
            if self.buffer[i].isspace():
                i += 1  # Skip whitespace
            elif self.buffer[i] == '{':
                # Start of a new object
                new_object = ASTNode(type='OBJECT', value=None, children=[])
                if self.current_object is None:
                    self.ast = new_object  # Root object
                else:
                    # Add the new object as a child of the current object
                    self.current_object.children.append(new_object)
                self.stack.append(self.current_object)
                self.current_object = new_object
                i += 1
            elif self.buffer[i] == '}':
                # End of the current object
                if self.stack:
                    self.current_object = self.stack.pop()
                i += 1
            elif self.buffer[i] == '"':
                # Parse a key or string value
                j = i + 1
                while j < n and self.buffer[j] != '"':
                    j += 1
                if j < n:
                    token = self.buffer[i + 1:j]  # Extract the token without quotes
                    i = j + 1
                    if self.current_object and self.buffer[i] == ':':
                        # This is a key
                        i += 1  # Consume ':'
                        # Parse the value
                        while i < n and self.buffer[i].isspace():
                            i += 1  # Skip whitespace
                        if i < n and self.buffer[i] == '"':
                            # String value
                            k = i + 1
                            while k < n and self.buffer[k] != '"':
                                k += 1
                            if k < n:
                                value = self.buffer[i + 1:k]
                                self.current_object.children.append(
                                    ASTNode(type='PAIR', value=token, children=[
                                        ASTNode(type='STRING', value=value, children=[])
                                    ])
                                )
                                i = k + 1
                        elif i < n and (self.buffer[i] == '-' or self.buffer[i].isdigit()):
                            # Integer value
                            k = i
                            while k < n and (self.buffer[k] == '-' or self.buffer[k].isdigit()):
                                k += 1
                            value = int(self.buffer[i:k])
                            self.current_object.children.append(
                                ASTNode(type='PAIR', value=token, children=[
                                    ASTNode(type='INTEGER', value=value, children=[])
                                ])
                            )
                            i = k
                    else:
                        # This is a string value (not part of a key-value pair)
                        self.current_object.children.append(
                            ASTNode(type='STRING', value=token, children=[])
                        )
                else:
                    # Incomplete string, wait for more input
                    break
            elif self.buffer[i] == ',':
                # Separator between key-value pairs
                i += 1
            else:
                # Parse an integer value
                if self.buffer[i] == '-' or self.buffer[i].isdigit():
                    j = i
                    while j < n and (self.buffer[j] == '-' or self.buffer[j].isdigit()):
                        j += 1
                    value = int(self.buffer[i:j])
                    self.current_object.children.append(
                        ASTNode(type='INTEGER', value=value, children=[])
                    )
                    i = j
                else:
                    # Unexpected character
                    i += 1
        self.buffer = self.buffer[i:]  # Remove processed part from buffer

    def process_input(self, new_input):
        """Process new input and update the AST."""
        self.tokenize_and_parse(new_input)

    def get_ast(self):
        """Get the current AST."""
        return self.ast

# Function to print the AST
def print_ast(node, indent=0):
    """Print the AST in a readable format."""
    if node is None:
        return
    print('  ' * indent + f'{node.type}: {node.value}')
    for child in node.children:
        print_ast(child, indent + 1)

# Example usage
parser = IncrementalJSONParser()

# Simulate batched input
batch1 = '{"name": "J'
batch2 = 'ohn", "age": 30, "address": { "city": "New York" }'

parser.process_input(batch1)
print("After Batch 1:")
print("Current AST:")
print_ast(parser.get_ast())

parser.process_input(batch2)
print("\nAfter Batch 2:")
print("Current AST:")
print_ast(parser.get_ast())

After Batch 1:
Current AST:
OBJECT: None

After Batch 2:
Current AST:
OBJECT: None
  STRING: John
  PAIR: age
    INTEGER: 30
  OBJECT: None
    PAIR: city
      STRING: New York


In [ ]:
import re
from collections import namedtuple

# Define a simple AST node structure
ASTNode = namedtuple('ASTNode', ['type', 'value', 'children'])

# Token types
TOKEN_TYPES = {
    'KEY': r'"[^"]+":',  # Keys must be in the format "key":
    'STRING_VALUE': r'"[^"]*"',  # Quoted strings
    'INTEGER_VALUE': r'-?\d+',  # Integers
    'OPEN_BRACE': r'\{',  # Start of a dictionary
    'CLOSE_BRACE': r'\}',  # End of a dictionary
    'WHITESPACE': r'\s+',  # Ignore whitespace
    'COMMA': r',',  # Separator between key-value pairs
}

# Stateful JSON parser
class IncrementalJSONParser:
    def __init__(self):
        self.buffer = ""  # Buffer to store incomplete input
        self.tokens = []  # List of tokens
        self.ast = None  # The resulting AST
        self.partial_key = None  # Track incomplete keys
        self.partial_value = None  # Track incomplete values
        self.current_object = None  # Track the current object being parsed

    def tokenize(self, new_input):
        """Tokenize the input and add to the buffer."""
        self.buffer += new_input
        token_regex = '|'.join(f'(?P<{name}>{pattern})' for name, pattern in TOKEN_TYPES.items())
        for match in re.finditer(token_regex, self.buffer):
            token_type = match.lastgroup
            token_value = match.group(token_type)
            if token_type != 'WHITESPACE':  # Ignore whitespace
                self.tokens.append((token_type, token_value))
                self.buffer = self.buffer[match.end():]  # Remove processed part from buffer

    def parse(self):
        """Parse the tokens into an AST incrementally."""
        while self.tokens:
            token_type, token_value = self.tokens[0]

            if token_type == 'OPEN_BRACE':
                # Start of a new object
                self.tokens.pop(0)  # Consume '{'
                new_object = ASTNode(type='OBJECT', value=None, children=[])
                if self.current_object is None:
                    self.ast = new_object  # Root object
                else:
                    # Add the new object as a child of the current object
                    self.current_object.children.append(new_object)
                self.current_object = new_object

            elif token_type == 'CLOSE_BRACE':
                # End of the current object
                self.tokens.pop(0)  # Consume '}'
                if self.current_object:
                    self.current_object = None  # Move back to the parent object

            elif token_type == 'KEY':
                # Parse a key-value pair
                self.tokens.pop(0)  # Consume the key
                key = token_value.strip('":')
                if self.tokens:
                    value_node = self.parse_value()
                    if value_node:
                        if self.current_object:
                            self.current_object.children.append(ASTNode(type='PAIR', value=key, children=[value_node]))

            elif token_type == 'COMMA':
                # Separator between key-value pairs
                self.tokens.pop(0)  # Consume ','

    def parse_value(self):
        """Parse a value (string, integer, or dictionary)."""
        if not self.tokens:
            return None
        token_type, token_value = self.tokens[0]
        if token_type == 'OPEN_BRACE':
            # Start of a nested object
            return None  # The object will be handled in the main parse loop
        elif token_type == 'STRING_VALUE':
            self.tokens.pop(0)
            return ASTNode(type='STRING', value=token_value.strip('"'), children=[])
        elif token_type == 'INTEGER_VALUE':
            self.tokens.pop(0)
            return ASTNode(type='INTEGER', value=int(token_value), children=[])
        return None

    def process_input(self, new_input):
        """Process new input and update the AST."""
        self.tokenize(new_input)
        self.parse()

    def get_ast(self):
        """Get the current AST."""
        return self.ast

    def get_partial_results(self):
        """Get partial results (incomplete keys or values)."""
        partial_results = {}
        if self.partial_key:
            partial_results["partial_key"] = self.partial_key
        if self.partial_value:
            partial_results["partial_value"] = self.partial_value
        return partial_results

# Function to print the AST
def print_ast(node, indent=0):
    """Print the AST in a readable format."""
    if node is None:
        return
    print('  ' * indent + f'{node.type}: {node.value}')
    for child in node.children:
        print_ast(child, indent + 1)

# Example usage
parser = IncrementalJSONParser()

# Simulate batched input
batch1 = '{"name": "J'
batch2 = 'ohn", "age": 30, "address": { "city": "New York" }'

parser.process_input(batch1)
print("After Batch 1:")
print("Partial Results:", parser.get_partial_results())
print("Current AST:")
print_ast(parser.get_ast())

parser.process_input(batch2)
print("\nAfter Batch 2:")
print("Partial Results:", parser.get_partial_results())
print("Current AST:")
print_ast(parser.get_ast())

After Batch 1:
Partial Results: {}
Current AST:
OBJECT: None


KeyboardInterrupt: 

In [ ]:
import re
from collections import namedtuple

# Define a simple AST node structure
ASTNode = namedtuple('ASTNode', ['type', 'value', 'children'])

# Token types
TOKEN_TYPES = {
    'KEY': r'"[^"]+":',
    'STRING_VALUE': r'"[^"]*"',
    'NUMBER_VALUE': r'-?\d+(\.\d+)?([eE][+-]?\d+)?',
    'BOOLEAN_VALUE': r'true|false',
    'NULL_VALUE': r'null',
    'UNQUOTED_VALUE': r'[^,}\]]+',
    'WHITESPACE': r'\s+',
    'COMMA': r',',
    'OPEN_BRACE': r'\{',
    'CLOSE_BRACE': r'\}',
    'OPEN_BRACKET': r'\[',
    'CLOSE_BRACKET': r'\]'
}

# Stateful JSON parser
class IncrementalJSONParser:
    def __init__(self):
        self.buffer = ""
        self.tokens = []
        self.current_state = "START"
        self.ast = None

    def tokenize(self, new_input):
        self.buffer += new_input
        token_regex = '|'.join(f'(?P<{name}>{pattern})' for name, pattern in TOKEN_TYPES.items())
        for match in re.finditer(token_regex, self.buffer):
            token_type = match.lastgroup
            token_value = match.group(token_type)
            if token_type != 'WHITESPACE':
                self.tokens.append((token_type, token_value))
                self.buffer = self.buffer[match.end():]  # Remove processed part from buffer

    def parse(self):
        def parse_value():
            if not self.tokens:
                return None
            token_type, token_value = self.tokens[0]
            if token_type == 'OPEN_BRACE':
                return parse_object()
            elif token_type == 'OPEN_BRACKET':
                return parse_array()
            elif token_type in ('STRING_VALUE', 'NUMBER_VALUE', 'BOOLEAN_VALUE', 'NULL_VALUE', 'UNQUOTED_VALUE'):
                self.tokens.pop(0)
                return ASTNode(type='VALUE', value=token_value, children=[])
            return None

        def parse_object():
            if not self.tokens or self.tokens[0][0] != 'OPEN_BRACE':
                return None
            self.tokens.pop(0)  # Consume '{'
            node = ASTNode(type='OBJECT', value=None, children=[])
            while self.tokens and self.tokens[0][0] != 'CLOSE_BRACE':
                key_token = self.tokens.pop(0)
                if key_token[0] != 'KEY':
                    raise SyntaxError(f"Expected key, got {key_token}")
                key = key_token[1].strip('":')
                if self.tokens and self.tokens[0][0] == 'COMMA':
                    self.tokens.pop(0)  # Consume ','
                value_node = parse_value()
                if value_node:
                    node.children.append(ASTNode(type='PAIR', value=key, children=[value_node]))
            if self.tokens and self.tokens[0][0] == 'CLOSE_BRACE':
                self.tokens.pop(0)  # Consume '}'
            return node

        def parse_array():
            if not self.tokens or self.tokens[0][0] != 'OPEN_BRACKET':
                return None
            self.tokens.pop(0)  # Consume '['
            node = ASTNode(type='ARRAY', value=None, children=[])
            while self.tokens and self.tokens[0][0] != 'CLOSE_BRACKET':
                value_node = parse_value()
                if value_node:
                    node.children.append(value_node)
                if self.tokens and self.tokens[0][0] == 'COMMA':
                    self.tokens.pop(0)  # Consume ','
            if self.tokens and self.tokens[0][0] == 'CLOSE_BRACKET':
                self.tokens.pop(0)  # Consume ']'
            return node

        self.ast = parse_object()

    def process_input(self, new_input):
        self.tokenize(new_input)
        self.parse()

    def get_ast(self):
        return self.ast

# Function to print the AST
def print_ast(node, indent=0):
    if node is None:
        return
    print('  ' * indent + f'{node.type}: {node.value}')
    for child in node.children:
        print_ast(child, indent + 1)

# Example usage
parser = IncrementalJSONParser()

# Simulate batched input
batch1 = '{"name": "J'
batch2 = 'ohn"}'

parser.process_input(batch1)
print("After Batch 1:")
print_ast(parser.get_ast())

parser.process_input(batch2)
print("\nAfter Batch 2:")
print_ast(parser.get_ast())

After Batch 1:

After Batch 2:


In [ ]:
import re

regex = r'(?<![:{,]\s*)"(.*?)"(?=\s*[},])|(?<![:{,]\s*)([^\s\",:{}]+)(?=\s*[},])'

texts = [
    '{"x": "oooo"}',  # Should match oooo (without quotes)
    '"xxxxx"',  # Should match xxxxx
    '"xxxx',    # Should match xxxx (incomplete string)
    'xxxx"',    # Should match xxxx (incomplete string)
    'xxx',      # Should match xxx
    '"hello world"',  # Should match hello world
    '"hello',         # Should match hello (incomplete)
    'world"',         # Should match world (incomplete)
    '{ "y": "value" }',  # Should match value
    ' { "key": "incomplete',  # Should match incomplete
    'another_value'  # Should match another_value
]

for text in texts:
    matches = [m for m in re.findall(regex, text) if any(m)]  # Filter empty matches
    print(f"Text: {text} -> Matches: {matches}")


error: look-behind requires fixed-width pattern

In [ ]:
import json

class IncrementalJsonParser:
    def __init__(self):
        self.buffer = ""

    def add_chunk(self, chunk):
        """Appends new data to the buffer and tries parsing."""
        self.buffer += chunk
        return self.try_parse()

    def try_parse(self):
        """Attempts to parse the buffered JSON."""
        try:
            data = json.loads(self.buffer)
            self.buffer = ""  # Clear buffer after successful parsing
            return data
        except json.JSONDecodeError:
            return None  # Parsing failed, waiting for more data

# Example usage
parser = IncrementalJsonParser()

# First batch (incomplete JSON)
print(parser.add_chunk('{"name": "Joh'))  # None (not yet complete)

# Second batch (completes the JSON)
#print(parser.add_chunk('n"}'))  # {"name": "John"} (successfully parsed)


None


In [ ]:
import json

class JsonParser:
    def __init__(self):
        self.buffer = ""
        self.tree = {}
        self.current_obj = self.tree
        self.key = None
        self.stack = []  # Stack for nested structures

    def parse_batch(self, batch):
        self.buffer += batch
        self._process_buffer()

    def _process_buffer(self):
        while self.buffer:
            if self.key is None:
                self._parse_key()
            else:
                self._parse_value()

    def _parse_key(self):
        if ':' in self.buffer:
            key_part, rest = self.buffer.split(':', 1)
            key_part = key_part.strip()

            if key_part.startswith('{'):
                key_part = key_part[1:].strip()

            if key_part.startswith('"') and key_part.endswith('"'):
                self.key = json.loads(key_part)
                self.buffer = rest.strip()

    def _parse_value(self):
        if self.buffer.startswith('{'):
            # Start a new dictionary
            new_obj = {}
            self.current_obj[self.key] = new_obj
            self.stack.append((self.current_obj, self.key))  # Save current context
            self.current_obj = new_obj
            self.key = None
            self.buffer = self.buffer[1:].strip()
        else:
            # Partial value parsing
            match = self.buffer.split('"', 2)
            if len(match) >= 3:
                self.current_obj[self.key] = match[1]  # Complete value
                self.buffer = match[2].strip()
                self.key = None
            else:
                # Incomplete value, keep in buffer
                return

    def get_parsing_tree(self):
        return self.tree

# Example usage:
parser = JsonParser()
parser.parse_batch('{"x": "k')
print(parser.get_parsing_tree())  # Should show {"x": "k"} (partially parsed)

parser.parse_batch('akkdf"}')
print(parser.get_parsing_tree())  # Should show {"x": "kakkdf"} (fully parsed)

parser.parse_batch('{"y": "j"}')
print(parser.get_parsing_tree())  # Should show {"x": "kakkdf", "y": "j"}

KeyboardInterrupt: 

In [ ]:
import re

# Regulärer Ausdruck zur Identifikation von Schlüssel-Formaten
pattern = r'\"([^\"]+)\"\s*:'

# Test-String
test_string = '''
{
    "name": "John
}
'''

# Suche nach allen passenden Schlüsseln
matches = re.findall(pattern, test_string)

# Ergebnisse ausgeben
print("Gefundene Schlüssel:", matches[0])


Gefundene Schlüssel: ['name']


In [ ]:
def is_json_object(json_string):
    if re.match(r'^\s*{', json_string):
        print("This is a JSON object.")
    else:
        print("This is not a JSON object.")
is_json_object('{"a":')

This is a JSON object.


In [ ]:
import re

pattern = r'"(.*)'  # Match content inside curly braces, without requiring a closing brace

data = '"'  # Incomplete string with an opening brace

match = re.search(pattern, data)

if match:
    print("Matched string:", match.group(0))  # Capture everything starting from the opening brace
    print("Matched content:", match.group(1))  # Content inside the braces
    print("End position:", match.end())
else:
    print("No match found.")

Matched string: "
Matched content: 
End position: 1


In [ ]:
class StreamingJSONParser:
    def __init__(self):
        self.state = {}
        self.current_key = None
        self.position = 0
        self.current_value = None
        self.buffer = ""
        self.key_pattern = r'"(.*?)":'
        self.string_value_pattern = r'"(.+)"'
        self.incomplete_string_pattern = r'"([^"].+)$'
        self.non_string_value_pattern = r'\d+'
        self.empty_string_value_pattern = r'""'

    def _parse_key(self, data):
        """ Extract the key using regex. """
        match = re.search(self.key_pattern, data)
        if match:
            return match.group(1), match.end()
        return None, 0

    def _parse_new_json(self, data):
        match = re.search(r'\{.*?\}', data)
        if match:
            return match.group(0), match.end()
        return None, 0

    def _parse_string(self, data):
        """ Extract the string value using regex, handling both complete and incomplete strings. """
        # Try to find a complete string first
        match = re.search(r'"(.*?)"(?![^"]*":)', data)
        if match:
            return match.group(1), match.end()  # Full string matched
        return None, 0

    def _parse_string_beginning(self, data):
        # Try to find a complete string first
        match = re.search(r'"([^"]*)', data)
        if match:
            return match.group(1), match.end()  # Full string matched
        return None, 0

    def _parse_incomplete_string(self, data):
        """ Extract the incomplete string value using regex. """
        match = re.search(self.incomplete_string_pattern, data)
        if match:
            return match.group(1), match.end()
        return None, 0

    def _parse_empty_string(self, data):
        """ Extract the empty string value using regex. """
        match = re.search(self.empty_string_value_pattern, data)
        if match:
            return match.end()
        return None, 0

    def _parse_non_string_value(self, data):
        """ Extract non-string values (numbers, booleans, null) using regex. """
        match = re.search(self.non_string_value_pattern, data)
        if match:
            return match.group(0), match.end()
        return None, 0

    def _parse_string_continuation(self, data):
        """ Extract the string value using regex, handling both complete and incomplete strings. """
        # Try to find a complete string first
        match = re.search(r'(?<!")\b[A-Za-z0-9]+\b(?!")', data)
        if match:
            return match.group(0), match.end()
        return None, 0

    def _parse_ending_string(self, data):
        """ Extract the string value using regex, incomplete strings. """
        # Try to find a complete string first
        print("data to parse_ending_string :",data)
        match = re.search(r'[A-Za-z0-9]+(?=")', data)
        if match:
            return match.group(0), match.end()
        return None, 0

    def _parse_string_ending(self,data):
        # Try to find a complete string first
        match = re.search(r'(.*)"', data)
        if match:
            return match.group(1), match.end()  # Full string matched
        return None, 0

    def convert_to_true_type(self, value):
      # Check if the value is an integer
      if value.isdigit():  # Only works for non-negative integers
          return int(value)

      # Check if the value is a float (this is a basic check, more robust checks can be added)
      try:
          return float(value)
      except ValueError:
          pass  # Not a float

      # If it's none of the above, assume it's a string
      return value


    def consume(self, chunk):
        """ Process incremental chunks of the input data. """
        self.buffer += chunk
        print("+++++++++++++++++++++++++ initial buffer "+ self.buffer+" +++++++++++++++++++++++++")
        while self.position < len(self.buffer):
            if self.current_key is None:
                # Parse key
                key, key_end = self._parse_key(self.buffer[self.position:])
                if key:
                    print("found key------>")
                    #self.state[key] = None
                    self.current_key = key
                    self.position += key_end
                else:
                    self.position += 1

            if self.current_value is not None and self.current_key is not None:
              value, value_end = self._parse_ending_string(self.buffer[self.position:])
              if value:
                self.state[self.current_key] += value
                self.current_key = None
                self.current_value = None
                self.position += value_end
                continue

              value, value_end = self._parse_string_continuation(self.buffer[self.position])
              if value:
                self.state[self.current_key] += value
                self.position += value_end
                continue

              value, value_end = self._parse_string_beginning(self.buffer[self.position:])
              if value:
                  # If the value is found, store it in the state and reset the key and value
                  self.state[self.current_key] = value
                  self.current_key = None
                  self.current_value = None
                  self.position += value_end
                  continue

            if self.current_value is None:

                value, value_end = self._parse_string_beginning(self.buffer[self.position:])
                if value:
                    print("parse string beginning")
                    # If the value is found, store it in the state and reset the key and value
                    self.state[self.current_key] = value
                    self.current_key = None
                    self.current_value = None
                    self.position += value_end
                    continue

                value, value_end = self._parse_incomplete_string(self.buffer[self.position:])
                if value:
                    print("parse incomplete string")
                    # If the value is found, store it in the state and reset the key and value
                    self.state[self.current_key] = value
                    self.current_value = value
                    self.position += value_end
                    continue

                value, value_end = self._parse_string(self.buffer[self.position:])
                if value:
                    print("parse string")
                    # If the value is found, store it in the state and reset the key and value
                    self.state[self.current_key] = value
                    self.current_key = None
                    self.current_value = None
                    self.position += value_end
                    continue

                value, value_end = self._parse_string_ending(self.buffer[self.position:])
                if value:
                    print("parse string ending")
                    # If the value is found, store it in the state and reset the key and value
                    self.state[self.current_key] = value
                    self.current_key = None
                    self.current_value = None
                    self.position += value_end
                    continue

            self.position += 1

        return self.state


    def get(self):
        """ Return the current state of the parsed JSON object. """
        return self.state


# Example usage:
parser = StreamingJSONParser()

"""# First chunk of data (incomplete string for "test")
parser.consume('{"test": ""')
print(parser.get())  # Output: {}

parser.consume('"hel')
print(parser.get())  # Output: {"test": "hel"}

# Second chunk of data (finishing "test" string and starting "country")
parser.consume('lo", "country": "Sw')
print(parser.get())  # Output: {"test": "hello"}

# Final chunk of data (completing the "country" string)
parser.consume('itzerland", "age": 30}')
print(parser.get())  # Output: {"test": "hello", "country": "Switzerland", "age": 30}"""
parser = StreamingJSONParser()
parser.consume('{"foo":"bar')
print(parser.get())
parser.consume('ba')
parser.consume('ri')
parser.consume('sta"')
print(parser.get())

#assert parser.get() == {"foo": "bar"}"""

+++++++++++++++++++++++++ initial buffer {"foo":"bar +++++++++++++++++++++++++
found key------>
parse string beginning
{'foo': 'bar'}
+++++++++++++++++++++++++ initial buffer {"foo":"barba +++++++++++++++++++++++++
+++++++++++++++++++++++++ initial buffer {"foo":"barbari +++++++++++++++++++++++++
+++++++++++++++++++++++++ initial buffer {"foo":"barbarista" +++++++++++++++++++++++++
parse string ending
{'foo': 'bar', None: 'ta'}


In [ ]:
def test_streaming_json_parser():
 parser = StreamingJSONParser()
 parser.consume('{"foo": "bar"}')
 print(parser.get())
 assert parser.get() == {"foo": "bar"}

def test_chunked_streaming_json_parser():
 parser = StreamingJSONParser()
 parser.consume('{"foo":')
 parser.consume('"bar')
 print(parser.get())
 #assert parser.get() == {"foo": "bar"}

def test_partial_streaming_json_parser():
 parser = StreamingJSONParser()
 parser.consume('{"foo": "bar')
 print(parser.get())
 assert parser.get() == {"foo": "bar"}

test_streaming_json_parser()
test_chunked_streaming_json_parser()
test_partial_streaming_json_parser()

+++++++++++++++++++++++++ initial buffer {"foo": "bar"} +++++++++++++++++++++++++
found key------>
parse string beginning
{'foo': 'bar"}'}


AssertionError: 